# Day 2.3 — Embeddings and Semantic Search
Word overlap failed because the answer uses different words. An **embedding** replaces overlap with
a numeric representation of meaning. We index the corpus twice — hash embedder, then trained
sentence-transformer — and ask both the identical question from 2.2.

### Step 1 — What a vector actually is

The hash embedder turns text into 512 numbers: each word is hashed to one position and counted
there, then the vector is scaled to length 1. Nothing is learned, and because the vectors have
length 1 their dot product *is* the cosine similarity.

In [ ]:
import hashlib, math

def normalize(vector):
    magnitude = math.sqrt(sum(value * value for value in vector)) or 1.0
    return [value / magnitude for value in vector]

class HashEmbedder:
    """Deterministic, offline, and honest about itself: keyword matching in vector form."""
    name = "hash"

    def __init__(self, dimensions=512):
        self.dimensions = dimensions

    def embed(self, texts):
        vectors = []
        for text in texts:
            vector = [0.0] * self.dimensions
            for token in tokenize(text):                       # one hashed position per word
                position = int.from_bytes(hashlib.sha256(token.encode()).digest()[:4], "big") % self.dimensions
                vector[position] += 1.0
            vectors.append(normalize(vector))
        return vectors

def dot(left, right):
    return sum(a * b for a, b in zip(left, right))

hash_embedder = HashEmbedder()
vector = hash_embedder.embed(["The controller stops charging at 45 degrees."])[0]
print("numbers per vector:", len(vector), "| positions used:", sum(1 for v in vector if v), "(one per distinct word)")

for left, right in [("Charging stops at 45 degrees.", "Charging stops at 45 degrees."),
                    ("Charging stops at 45 degrees.", "The controller stops charging at 45 degrees."),
                    ("What keeps running during a blackout?", "Priority 1 loads include emergency lighting.")]:
    a, b = hash_embedder.embed([left, right])
    print(f"  similarity {dot(a, b):.3f}   {left!r} vs {right!r}")
print("1.0 identical, shared words in between, and the paraphrase 0.0 - no meaning anywhere.")

### Step 2 — An index: embed once, search many times

`VectorIndex` embeds every chunk once; `search` embeds the question, sorts by similarity, and
returns `Retrieved` objects — chunk, score, rank — because evaluation refers to the rank later.

In [ ]:
class Retrieved(BaseModel):
    chunk: Chunk
    score: float
    rank: int                                    # 1 = closest

class VectorIndex:
    def __init__(self, embedder):
        self.embedder, self.chunks, self.vectors = embedder, [], []

    def add(self, chunks):
        self.chunks.extend(chunks)
        self.vectors.extend(self.embedder.embed([chunk.searchable_text for chunk in chunks]))

    def search(self, query, top_k=3):
        query_vector = self.embedder.embed([query])[0]
        ranked = sorted(zip(self.chunks, self.vectors), key=lambda pair: dot(query_vector, pair[1]), reverse=True)
        return [Retrieved(chunk=chunk, score=dot(query_vector, vector), rank=rank)
                for rank, (chunk, vector) in enumerate(ranked[:top_k], start=1)]

def show(index, label, question, k=3):
    print(f"[{label}] {question}")
    for item in index.search(question, top_k=k):
        print(f"   rank {item.rank}  score {item.score:.3f}  {item.chunk.chunk_id:42} {item.chunk.section}")

def rank_of(index, question, chunk_id):
    """Where one chunk lands in the FULL ranking (1 = best)."""
    return next(item.rank for item in index.search(question, top_k=len(index.chunks))
                if item.chunk.chunk_id == chunk_id)

hash_index = VectorIndex(hash_embedder)
hash_index.add(chunks)
show(hash_index, "hash", PARAPHRASE)
print("\nrank of", EXPECTED_CHUNK, "->", rank_of(hash_index, PARAPHRASE, EXPECTED_CHUNK), "of", len(chunks))
print("The hash embedder is word overlap in a costume, so it fails the paraphrase exactly as 2.2 did.")

### Step 3 — Load a trained embedder

A sentence-transformer was trained on text where *blackout* and *islanded operation* co-occur. The
first load downloads about 90 MB; if it fails we fall back to the hash embedder and say so.

In [ ]:
%pip install -q sentence-transformers
print("sentence-transformers requested. If that failed, the next cell falls back to the hash embedder.")

In [ ]:
class SentenceTransformerEmbedder:
    """Local semantic embeddings, downloaded once and cached."""
    name = "semantic"

    def __init__(self, model_name="sentence-transformers/all-MiniLM-L6-v2"):
        from sentence_transformers import SentenceTransformer
        self.model = SentenceTransformer(model_name)

    def embed(self, texts):
        return [vector.tolist() for vector in self.model.encode(texts, normalize_embeddings=True)]

semantic_index = None
try:
    semantic_index = VectorIndex(SentenceTransformerEmbedder())
    semantic_index.add(chunks)
    print("Semantic index built with", len(semantic_index.chunks), "chunks.")
except Exception as error:                                   # no package, no network, corrupt cache
    print("Semantic embedder unavailable:", type(error).__name__, "-", str(error).splitlines()[0][:90])
    print("Falling back to the hash embedder. Every cell still runs; paraphrases retrieve worse, and")
    print("the comparisons below print 'not available' instead of pretending.")

BEST_INDEX = semantic_index or hash_index                    # the rest of the day uses the best we have
print("Best index available:", type(BEST_INDEX.embedder).__name__)

### Step 4 — The same question through three retrievers

One question, three representations. Only the trained model connects *blackout* with *islanded
operation*, and nothing about the chunks or the question changed.

In [ ]:
def keyword_rank(question, chunk_id):
    return [chunk.chunk_id for _, chunk in keyword_ranking(question)].index(chunk_id) + 1

print("Question:", PARAPHRASE, "\n")
print(f"{'retriever':22}{'rank of expected chunk':26}top-1 chunk")
print("-" * 92)
print(f"{'keyword overlap':22}{keyword_rank(PARAPHRASE, EXPECTED_CHUNK):<26}{keyword_ranking(PARAPHRASE)[0][1].chunk_id}")
print(f"{'hash embedder':22}{rank_of(hash_index, PARAPHRASE, EXPECTED_CHUNK):<26}{hash_index.search(PARAPHRASE, top_k=1)[0].chunk.chunk_id}")
if semantic_index is not None:
    print(f"{'semantic embedder':22}{rank_of(semantic_index, PARAPHRASE, EXPECTED_CHUNK):<26}{semantic_index.search(PARAPHRASE, top_k=1)[0].chunk.chunk_id}")
    print()
    show(semantic_index, "semantic", PARAPHRASE)
else:
    print(f"{'semantic embedder':22}{'not available':26}-")

### Step 5 — A score is a ranking, not a verdict

Similarity answers "which candidate is closest?", never "is this passage sufficient?". Ask
something the corpus cannot answer and watch the top score stay comfortably positive.

In [ ]:
for label, index in [("hash", hash_index)] + ([("semantic", semantic_index)] if semantic_index else []):
    for question in ["How long are battery fault records retained?",     # answerable
                     "What is the purchase price of the battery?"]:      # not in the corpus at all
        best = index.search(question, top_k=1)[0]
        print(f"[{label:9}] best score {best.score:.3f} -> {best.chunk.chunk_id:42} {question}")
print()
print("1. the unanswerable question never scores 0 - the closest chunk is always returned;")
print("2. the two embedders produce different numbers, so a cut-off tuned for one is meaningless for")
print("   the other. Sufficiency is decided in 2.5 and measured in 2.6, never by a threshold.")

### Step 6 — Optional: the same vectors inside Chroma

Our in-memory index keeps the mathematics visible; a vector database adds persistence, filtering
and scale. Nothing else today needs it, so skipping this cell is fine.

In [ ]:
try:
    import chromadb
    collection = chromadb.Client().get_or_create_collection("day2_lab", metadata={"hnsw:space": "cosine"})
    collection.upsert(ids=[chunk.chunk_id for chunk in chunks],
                      documents=[chunk.text for chunk in chunks],
                      embeddings=BEST_INDEX.embedder.embed([chunk.searchable_text for chunk in chunks]))
    hits = collection.query(query_embeddings=BEST_INDEX.embedder.embed([PARAPHRASE]), n_results=3)
    for rank, (chunk_id, distance) in enumerate(zip(hits["ids"][0], hits["distances"][0]), start=1):
        print(f"   rank {rank}  score {1 - distance:.3f}  {chunk_id}")
    print("Same embedder, same chunks - only the storage layer changed.")
except Exception as error:
    print("Chroma not available:", type(error).__name__, "- skipping.")
    print("Optional: run `%pip install -q chromadb` and re-run. Nothing else in Day 2 needs it.")

### Try it yourself

The Errors section says a rejected command must not be repeated. Ask about that in everyday words —
no *error*, no *retry*, no *command* — and predict which retriever finds it.

In [ ]:
# --- Worked solution ---------------------------------------------------------------
MY_QUESTION = "What happens if someone presses the wrong button twice?"
TARGET = "controller_interface:errors"

print("Question:", MY_QUESTION)
print("hash rank    :", rank_of(hash_index, MY_QUESTION, TARGET), "of", len(chunks))
if semantic_index is not None:
    print("semantic rank:", rank_of(semantic_index, MY_QUESTION, TARGET), "of", len(chunks))
    show(semantic_index, "semantic", MY_QUESTION)
else:
    print("semantic rank: not available in this environment")
print()
print("The section never says 'button' or 'twice'; it says 'Repeating a rejected operational command")
print("... is prohibited'. Word overlap cannot bridge that gap.")

### Checkpoint

**1. Which component creates vectors, and which one stores and searches them?**

<details><summary>Show answer</summary>

The *embedder* creates vectors; the *index* stores and ranks them. They are separate on purpose: change the storage without touching the embedder, or swap the embedder without touching the storage — the experiment in Step 4.

</details>

**2. The unanswerable question still scored well above zero. Could we reject everything below a fixed cut-off such as 0.5?**

<details><summary>Show answer</summary>

No. Scores depend on the model, the text length and the query, and are not calibrated probabilities: a threshold tuned on three questions breaks on the fourth. Decide with evidence instead — 2.5 and 2.6.

</details>

### Recap

- **Limitation seen:** word overlap, hash embedder included, cannot connect *blackout* with *islanded operation*.
- **Layer added:** a trained embedding model behind the same `VectorIndex`, with a printed fallback.
- **Evidence:** on the same question the expected chunk moves from rank 14 to rank 1.